# RAG Optimization + LLM Evaluation + LangSmith 
**🎥 LIVE CODING DEMO**

Build production-ready RAG systems with advanced optimization techniques, evaluation frameworks, and observability.

## 📋 Course Agenda:
1. ✅ **Multi-query Retriever** (20 min) - Multiple perspectives on user queries
2. ✅ **Context Compression Retriever** (20 min) - Filter irrelevant context
3. ✅ **LLM Evaluation - Faithfulness & Relevance** (20 min) - Quality metrics
4. ✅ **LLM-as-a-Judge Evaluation** (20 min) - Automated grading
5. ✅ **LangSmith Setup & Tracing** (20 min) - Production observability
6. ✅ **Debugging Bad RAG Responses** (20 min) - Diagnosis & improvement

In [3]:
# ============================================================================
# STEP 1: ENVIRONMENT SETUP - Load API Keys
# ============================================================================
# Purpose: Securely load OpenAI API key from environment file

# TODO: Import os module
# TODO: Import load_dotenv and find_dotenv from dotenv
# TODO: Load environment variables using load_dotenv(find_dotenv(), override=True)
# TODO: Get API key using os.getenv("OPENAI_API_KEY")
# TODO: Print first 10 chars of API key to verify it loaded

import os
from dotenv import load_dotenv, find_dotenv

# Load the .env file
load_dotenv(find_dotenv(), override=True)

# Verify the API key is loaded (show first 10 chars only for security)
api_key = os.getenv("OPENAI_API_KEY")
if api_key:
    print(f"✅ API Key loaded: {api_key[:10]}...")
else:
    print("❌ API Key not found! Check your .env file")

✅ API Key loaded: sk-proj-3n...


In [4]:
# ============================================================================
# STEP 2: PACKAGE INSTALLATION (if needed)
# ============================================================================
# Uncomment and run these if packages aren't installed:

# !pip install langchain langchain-community langchain-openai langchain-text-splitters
# !pip install pypdf faiss-cpu chromadb
# !pip install scikit-learn numpy pandas

# Part 1: Multi-Query Retriever (20 min)

## 🎯 Problem:
Single queries miss relevant documents. Different phrasings capture different aspects.

## ✅ Solution:
Generate 3-5 alternative questions → retrieve for each → deduplicate

In [5]:
# ============================================================================
# STEP 3: CORE IMPORTS
# ============================================================================

# TODO: Import ChatOpenAI, OpenAIEmbeddings from langchain_openai
# TODO: Import Chroma from langchain_community.vectorstores
# TODO: Import PyPDFLoader from langchain_community.document_loaders
# TODO: Import RecursiveCharacterTextSplitter from langchain_text_splitters

# TODO: Initialize ChatOpenAI(model="gpt-4o-mini", temperature=0.3) as 'llm'

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

print("✅ All core imports successful!")

# Create LLM for multi-query generation
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.3)
print("✅ LLM initialized for multi-query generation")

d:\Mentoring\learwithsarvesh\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ All core imports successful!
✅ LLM initialized for multi-query generation


In [6]:
# ============================================================================
# STEP 4: LOAD AND PREPARE DOCUMENTS
# ============================================================================

# TODO: Set 
pdf_path = "dataset.pdf"

# TODO: Create PyPDFLoader and load documents

# TODO: Create RecursiveCharacterTextSplitter:
#       chunk_size=500, chunk_overlap=100
if os.path.exists(pdf_path):
    loader = PyPDFLoader(pdf_path)
    documents = loader.load()

# TODO: Split documents into chunks
    text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap = 100,
    length_function = len,
)
    chunks = text_splitter.split_documents(documents)

# TODO: Create OpenAIEmbeddings()
    embeddings = OpenAIEmbeddings()
    vectorstore = Chroma.from_documents(
    chunks,
    embeddings,
    persist_directory= "./chroma_db_multi_query"
    )

# TODO: Create Chroma vectorstore with Chroma.from_documents()
#       persist_directory="./chroma_db_multi_query"

In [7]:
# ============================================================================
# STEP 5: BUILD MULTI-QUERY RETRIEVER
# ============================================================================

base_retriever = vectorstore.as_retriever(search_kwargs = {"k":5})

# TODO: Import ChatPromptTemplate, StrOutputParser from langchain_core
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# TODO: Create base_retriever with vectorstore.as_retriever(search_kwargs={"k": 5})

# TODO: Create multi_query_prompt to generate {num_queries} alternatives for {question}
multi_query_prompt = ChatPromptTemplate.from_template(
    """You are an AI language model assistant. Your task is to generate {num_queries} different versions of the given user question to retrieve relevant documents from a vector database.
    Generate {num_queries} alternative questions that cover the same topic but with different wording and perspectives.
    Original question: {question}
    
    Provide only the questions, one per line, without numbering or bullet points."""
)


# TODO: Define class SimpleMultiQueryRetriever:
#       __init__(base_retriever, llm)
#       invoke(query, num_queries=3):
#           - Generate alternatives using LLM chain
#           - Retrieve for each query
#           - Deduplicate using set()
#           - Return unique documents

class SimpleMultiQueryRetriever:
    def __init__(self, base_retriever, llm):
        self.base_retriever = base_retriever
        self.llm = llm

    def invoke(self, query: str, num_queries: int = 3):
        # Generate alternative questions
        chain = multi_query_prompt | self.llm | StrOutputParser()
        questions_text = chain.invoke({"question": query, "num_queries": num_queries})
        questions = [q.strip() for q in questions_text.split('\n') if q.strip()]

        # Retrieve documents for each question (original + alternatives)
        all_docs = []
        
        # First, retrieve for the original query
        docs = self.base_retriever.invoke(query)
        all_docs.extend(docs)
        
        # Then retrieve for each alternative query
        for q in questions[:num_queries]:
            docs = self.base_retriever.invoke(q)
            all_docs.extend(docs)
        
        # Deduplicate based on content
        seen_content = set()
        unique_docs = []
        for doc in all_docs:
            if doc.page_content not in seen_content:
                seen_content.add(doc.page_content)
                unique_docs.append(doc)
        
        return unique_docs


multi_query_retriever = SimpleMultiQueryRetriever(base_retriever, llm)


In [8]:
# ============================================================================
# STEP 6: DEMO - Multi-Query Breakdown
# ============================================================================

# TODO: Set 
test_query = "What are the PAN card requirements and details?"

# STEP 6.1: Generate alternative queries
# TODO: Create prompt and chain to generate 3 alternatives
# TODO: Print all alternatives
alternatives_prompt = ChatPromptTemplate.from_template("""You are an AI language model assistant. Your task is to generate three 
different versions of the given user question to retrieve relevant documents from a vector database. Provide these alternative questions separated by newlines.

Original question: {question}""")

llm_for_alternatives = ChatOpenAI(model = "gpt-4o-mini", temperature=0.3, api_key = api_key)
alternatives_chain = alternatives_prompt | llm_for_alternatives | StrOutputParser()

alternatives_response = alternatives_chain.invoke({"question": test_query})
alternatives_list = [q.strip() for q in alternatives_response.split('\n') if q.strip() and q.strip() != test_query][:3]

print(f"\n✨ Generated {len(alternatives_list)} alternative questions:\n")
for i, alt_query in enumerate(alternatives_list, 1):
    print(f"   {i}. {alt_query}")

# STEP 6.2: Retrieve for each query
# TODO: Retrieve for original + each alternative
# TODO: Print count for each

all_retrieved_docs = []
doc_ids = set()

print(f"\n🔹 Original Query ({test_query}):")
docs_original = base_retriever.invoke(test_query)

all_retrieved_docs.extend(docs_original)

for doc in docs_original:
    doc_id = id(doc)
    doc_ids.add(doc_id)
print(f"   ✓ Retrieved {len(docs_original)} documents")

for i, alt_query in enumerate(alternatives_list, 1):
    print(f"\n🔹 Alternative {i} ({alt_query}):")
    docs_alt = base_retriever.invoke(alt_query)
    all_retrieved_docs.extend(docs_alt)
    print(f"   ✓ Retrieved {len(docs_alt)} documents")


# STEP 6.3: Deduplicate
# TODO: Use hash(doc.page_content) to find duplicates
# TODO: Print stats: total retrieved, unique, duplicates removed

unique_docs = []
seen_content = set()

for doc in all_retrieved_docs:
    content_hash = hash(doc.page_content)
    if content_hash not in seen_content:
        unique_docs.append(doc)
        seen_content.add(content_hash)

print(f"\n📊 Deduplication Results:")
print(f"   Total docs retrieved: {len(all_retrieved_docs)}")
print(f"   Unique documents: {len(unique_docs)}")
print(f"   Duplicates removed: {len(all_retrieved_docs) - len(unique_docs)}")


# STEP 6.4: Display results
# TODO: Show first 3 documents with metadata
print("\n" + "="*80)
print("STEP 4: FINAL RESULTS")
print("="*80)

print(f"\n✅ Final Unique Documents Retrieved: {len(unique_docs)}\n")

for i, doc in enumerate(unique_docs[:5], 1):  # Show first 5
    source = doc.metadata.get('source', 'Unknown')
    page = doc.metadata.get('page', 'Unknown')
    print(f"📄 Document {i}:")
    print(f"   Source: {source}")
    print(f"   Page: {page}")
    print(f"   Content Preview: {doc.page_content[:150]}...")
    print()



✨ Generated 3 alternative questions:

   1. What are the requirements and details for obtaining a PAN card?
   2. Can you provide information on the necessary criteria and details for a PAN card?
   3. What do I need to know about the requirements and specifics of a PAN card?

🔹 Original Query (What are the PAN card requirements and details?):
   ✓ Retrieved 5 documents

🔹 Alternative 1 (What are the requirements and details for obtaining a PAN card?):
   ✓ Retrieved 5 documents

🔹 Alternative 2 (Can you provide information on the necessary criteria and details for a PAN card?):
   ✓ Retrieved 5 documents

🔹 Alternative 3 (What do I need to know about the requirements and specifics of a PAN card?):
   ✓ Retrieved 5 documents

📊 Deduplication Results:
   Total docs retrieved: 20
   Unique documents: 3
   Duplicates removed: 17

STEP 4: FINAL RESULTS

✅ Final Unique Documents Retrieved: 3

📄 Document 1:
   Source: dataset.pdf
   Page: 4
   Content Preview: weeks.  
 
### Documents requi

In [9]:
# ============================================================================
# STEP 7: COMPARISON - Basic vs Multi-Query
# ============================================================================

# TODO: Test with query = "What benefits are covered?"
# TODO: Compare document counts from basic_retriever vs multi_query_retriever

# Compare: Basic Retriever vs MultiQueryRetriever

print("="*70)
print("COMPARISON: BASIC vs MULTI-QUERY RETRIEVER")
print("="*70)

query = "What documents are needed for PAN card application?"

# Basic retriever
basic_results = base_retriever.invoke(query)

# Multi-query retriever
multi_results = multi_query_retriever.invoke(query)

print(f"\n📊 Query: '{query}'")
print(f"\n🔹 Basic Retriever: {len(basic_results)} documents retrieved")
print(f"🔹 Multi-Query Retriever: {len(multi_results)} documents retrieved")
print(f"\n✅ Multi-Query usually retrieves MORE documents")
print(f"   (by considering multiple query reformulations)")

COMPARISON: BASIC vs MULTI-QUERY RETRIEVER

📊 Query: 'What documents are needed for PAN card application?'

🔹 Basic Retriever: 5 documents retrieved
🔹 Multi-Query Retriever: 2 documents retrieved

✅ Multi-Query usually retrieves MORE documents
   (by considering multiple query reformulations)


# Part 2: Context Compression 

## 🎯 Problem:
Retrieved docs contain noise → wastes tokens, slows generation, confuses model

## ✅ Solution:
Use LLM to extract ONLY relevant excerpts → 50-90% token savings

In [10]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

print("✅ Context Compression imports successful!")

# Simple prompt to extract only relevant information
compression_prompt = ChatPromptTemplate.from_template(
    """Extract only the relevant information from this document that answers the question.

Question: {question}

Document:
{document}

Relevant excerpt:"""
)

# Create a simple context compressor using the LLM
# This will extract only relevant parts from documents
class ContextCompressor:
    """
    Simple context compressor that extracts relevant parts from documents.
    """
    def __init__(self, base_retriever, llm):
        self.base_retriever = base_retriever
        self.llm = llm
        self.compress_chain = compression_prompt | llm | StrOutputParser()
    
    def invoke(self, query: str):
        # Retrieve documents
        docs = self.base_retriever.invoke(query)
        compressed_docs = []
        
        for doc in docs:
            # Compress the document
            compressed_content = self.compress_chain.invoke({
                "question": query,
                "document": doc.page_content[:500]
            })
            
            # Update document with compressed content
            doc.page_content = compressed_content
            compressed_docs.append(doc)
        
        return compressed_docs
    
    # Create compression retriever
compression_retriever = ContextCompressor(base_retriever, llm)

print("✅ Context Compression Retriever created!")
print("\nHow it works:")
print("  1. Retrieve documents")
print("  2. Extract only relevant parts using LLM")
print("  3. Return compressed documents")
print("  5. Reduce token usage and noise")

✅ Context Compression imports successful!
✅ Context Compression Retriever created!

How it works:
  1. Retrieve documents
  2. Extract only relevant parts using LLM
  3. Return compressed documents
  5. Reduce token usage and noise


In [14]:
# Demo: Context Compression
query = "What information is required to apply for a PAN card?"

print("="*70)
print("CONTEXT COMPRESSION DEMO")
print("="*70)
print(f"\n🔍 Query: {query}\n")


# Get results with compression
compressed_docs = compression_retriever.invoke(query)

print(f"✅ Retrieved and compressed {len(compressed_docs)} documents\n")
print("="*70)
print("COMPRESSED CONTENT:")
print("="*70)

# loop
# Get compressed results
compressed_docs = compression_retriever.invoke(query)

print(f"✅ Retrieved and compressed {len(compressed_docs)} documents\n")
print("="*70)
print("COMPRESSED CONTENT:")
print("="*70)

for i, doc in enumerate(compressed_docs[:2], 1):
    print(f"\n📄 Document {i}:")
    print("─"*70)
    print(f"Compressed size: {len(doc.page_content)} characters")
    print(f"\nContent:")
    print(doc.page_content)
    print("─"*70)





CONTEXT COMPRESSION DEMO

🔍 Query: What information is required to apply for a PAN card?

✅ Retrieved and compressed 5 documents

COMPRESSED CONTENT:
✅ Retrieved and compressed 5 documents

COMPRESSED CONTENT:

📄 Document 1:
──────────────────────────────────────────────────────────────────────
Compressed size: 295 characters

Content:
To apply for a PAN card, the following information is required:

- Copy of Existing PAN card
- Passport (Any Country) / OCI Card
- Passport Size Photograph
- Overseas address proof with zip code (Supporting documents - Indian NRO/NRE Account statement or Overseas bank statement or Utility bill)
──────────────────────────────────────────────────────────────────────

📄 Document 2:
──────────────────────────────────────────────────────────────────────
Compressed size: 294 characters

Content:
To apply for a PAN card, the following documents are required:

- Copy of Existing PAN card
- Passport (Any Country) / OCI Card
- Passport Size Photograph
- Overseas a

In [15]:
# Comparison: Raw vs Compressed
print("\n" + "="*70)
print("RAW vs COMPRESSED COMPARISON")
print("="*70)

# Raw retrieval (NO compression)
raw_docs = base_retriever.invoke(query)
raw_tokens = sum(len(doc.page_content.split()) for doc in raw_docs)

# Compressed retrieval
compressed_docs = compression_retriever.invoke(query)
compressed_tokens = sum(len(doc.page_content.split()) for doc in compressed_docs)

# Calculate the reduction:
reduction = ((raw_tokens - compressed_tokens) / raw_tokens * 100) if raw_tokens > 0 else 0

# Comparison: Raw vs Compressed - Token Savings
print(f"\n📊 Query: '{query}'")
print(f"\n🔹 Raw Retrieval (No Compression):")
print(f"   - Documents: {len(raw_docs)}")
print(f"   - Approx tokens: {raw_tokens}")
print(f"\n🔹 Compressed Retrieval:")
print(f"   - Documents: {len(compressed_docs)}")
print(f"   - Approx tokens: {compressed_tokens}")
print(f"\n💡 Token Reduction: {reduction:.1f}%")
print(f"💰 Cost Savings: {reduction:.1f}% fewer tokens sent to LLM")
print(f"⚡ Speed Improvement: Faster processing with less context")



RAW vs COMPRESSED COMPARISON

📊 Query: 'What information is required to apply for a PAN card?'

🔹 Raw Retrieval (No Compression):
   - Documents: 5
   - Approx tokens: 380

🔹 Compressed Retrieval:
   - Documents: 5
   - Approx tokens: 245

💡 Token Reduction: 35.5%
💰 Cost Savings: 35.5% fewer tokens sent to LLM
⚡ Speed Improvement: Faster processing with less context


# Part 3: LLM Evaluation - Faithfulness & Relevance

## 🎯 Why Evaluation Matters:
- **Faithfulness**: Is the answer grounded in retrieved documents? (No hallucinations!)
- **Relevance**: Does the answer address the user's question?
- **Both metrics needed** for quality RAG systems

## 📝 What We'll Build:
Simple evaluation system to detect:
1. ❌ Hallucinations (Response B)
2. ❌ Off-topic answers (Response C)
3. ✅ Good responses (Response A)

## 💡 Simple Example: France Capital Question

**Test Scenario:**
- Question: "What is the capital of France?"
- Context: "Paris is the capital and largest city of France."
- 3 Responses to evaluate (Good, Hallucination, Off-topic)

In [1]:
# ============================================================================
# STEP 10: Setup Test Data for Evaluation
# ============================================================================
# Recording Hint: Explain the 3 test cases - Good, Hallucination, Off-topic

# TODO: Import ChatPromptTemplate and StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser


# TODO: Define test question and context
question = "What is the capital of France?"
context = "Paris is the capital and largest city of France."


# TODO: Define three responses to evaluate:
# Response A: Perfect answer (faithful + relevant)
# Response B: Hallucination (adds population not in context)
# Response C: Off-topic (doesn't answer the question)
response_a = "The capital of France is Paris."
response_b = "The capital of France is Paris, which has a population of 2.1 million."
response_c = "France is a country in Europe."

# TODO: Print the test setup
# Hint: Show question, context, and all 3 responses
print("="*70)
print("SIMPLE EVALUATION EXAMPLE")
print("="*70)
print(f"\n📝 Question: {question}")
print(f"📄 Context: {context}\n")
print("Responses to evaluate:")
print(f"  A: {response_a}")
print(f"  B: {response_b}")
print(f"  C: {response_c}")
print("\n" + "="*70)

d:\Mentoring\learwithsarvesh\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


SIMPLE EVALUATION EXAMPLE

📝 Question: What is the capital of France?
📄 Context: Paris is the capital and largest city of France.

Responses to evaluate:
  A: The capital of France is Paris.
  B: The capital of France is Paris, which has a population of 2.1 million.
  C: France is a country in Europe.



### Step 1: Evaluate FAITHFULNESS 🔍

**Goal**: Check if answer uses ONLY information from the context (no hallucinations)

**Scoring**:
- 10 = All info from context ✅
- 5 = Some info not in context ⚠️
- 0 = Contradicts or adds external info ❌

In [2]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

llm = ChatOpenAI(model = 'gpt-4o-mini', temperature = 0.3)
print("LLM initialized for evaluation")

LLM initialized for evaluation


In [3]:
# ============================================================================
# STEP 11: Faithfulness Evaluation
# ============================================================================
# Recording Hint: Explain that faithfulness detects hallucinations

# TODO: Create a ChatPromptTemplate for faithfulness evaluation
# Hint: Template should ask "Is the answer ONLY using information from the context?"
# Hint: Define scoring: 10 (all from context), 5 (partial), 0 (contradicts/adds info)
# Hint: Ask for SCORE and REASON in response
faithfulness_prompt = ChatPromptTemplate.from_template("""
You are an evaluation expert. Check if the answer is faithful to the context.

Context: {context}
Answer: {answer}

Is the answer ONLY using information from the context?
- If YES (all info from context): Score 10
- If PARTIAL (some info not in context): Score 5
- If NO (contradicts or adds external info): Score 0

Respond ONLY with:
SCORE: [number]
REASON: [one sentence]
""")

# TODO: Create evaluation chain
# Hint: faithfulness_prompt | llm | StrOutputParser()

faithfulness_chain = faithfulness_prompt | llm | StrOutputParser()


# TODO: Print header (1️⃣ FAITHFULNESS EVALUATION)


# TODO: Loop through all 3 responses and evaluate each one
# Hint: Use [("A", response_a), ("B", response_b), ("C", response_c)]
# Hint: Invoke chain with context and answer
# Hint: Print response and result
for label, response in [("A", response_a), ("B", response_b), ("C", response_c)]:
    result = faithfulness_chain.invoke({
        "context": context,
        "answer": response
    })
    print(f"\nResponse {label}: \"{response}\"")
    print(result)
    print("-"*70)



Response A: "The capital of France is Paris."
SCORE: 10  
REASON: The answer directly restates the information provided in the context without adding or contradicting any details.
----------------------------------------------------------------------

Response B: "The capital of France is Paris, which has a population of 2.1 million."
SCORE: 5  
REASON: The answer correctly identifies Paris as the capital of France but includes an unverified population figure not mentioned in the context.
----------------------------------------------------------------------

Response C: "France is a country in Europe."
SCORE: 5  
REASON: The answer contains information that is not present in the context, specifically the geographical classification of France as a country in Europe.
----------------------------------------------------------------------


### Step 2: Evaluate RELEVANCE 🎯

**Goal**: Check if answer directly addresses the question

**Scoring**:
- 10 = Directly answers question ✅
- 5 = Related but incomplete ⚠️
- 0 = Doesn't answer the question ❌

In [4]:
# ============================================================================
# STEP 12: Relevance Evaluation
# ============================================================================
# Recording Hint: Explain that relevance detects off-topic answers

# TODO: Create relevance evaluation prompt
# Hint: Template should ask "Does the answer directly address what was asked?"
# Hint: Same scoring structure as faithfulness (10/5/0)
# Hint: Pass {question} and {answer} as template variables
relevance_prompt = ChatPromptTemplate.from_template("""
You are an evaluation expert. Check if the answer is relevant to the question.

Question: {question}
Answer: {answer}

Does the answer directly address what was asked?
- If YES (directly answers): Score 10
- If PARTIAL (related but incomplete): Score 5
- If NO (doesn't answer the question): Score 0

Respond ONLY with:
SCORE: [number]
REASON: [one sentence]
""")

# TODO: Create relevance chain
# Hint: relevance_prompt | llm | StrOutputParser()
relevance_chain = relevance_prompt | llm | StrOutputParser()

# TODO: Print header (2️⃣ RELEVANCE EVALUATION)


# TODO: Loop through responses and evaluate relevance
# Hint: Invoke chain with question and answer (NOT context!)
# Hint: Print response and result
for label, response in [("A", response_a), ("B", response_b), ("C", response_c)]:
    result = relevance_chain.invoke({
        "question": question,
        "answer": response
    })
    print(f"\nResponse {label}: \"{response}\"")
    print(result)
    print("-"*70)


Response A: "The capital of France is Paris."
SCORE: 10  
REASON: The answer directly states that the capital of France is Paris, which fully addresses the question.
----------------------------------------------------------------------

Response B: "The capital of France is Paris, which has a population of 2.1 million."
SCORE: 10  
REASON: The answer directly states that the capital of France is Paris, which fully addresses the question.
----------------------------------------------------------------------

Response C: "France is a country in Europe."
SCORE: 0  
REASON: The answer does not provide the capital of France, which is what was asked.
----------------------------------------------------------------------


### Step 3: Combined Evaluation (Faithfulness + Relevance) ⚡

**Goal**: Evaluate both dimensions together with overall verdict

**Verdict**: PASS if both scores >= 8

In [5]:
# ============================================================================
# STEP 13: Combined Evaluation (Faithfulness + Relevance)
# ============================================================================
# Recording Hint: Show how to evaluate both dimensions together with PASS/FAIL verdict

# TODO: Create combined evaluation prompt
# Hint: Evaluate BOTH faithfulness AND relevance in single prompt
# Hint: Pass {question}, {context}, and {answer}
# Hint: Return FAITHFULNESS score, RELEVANCE score, OVERALL, VERDICT (PASS if both >= 8)
combined_prompt = ChatPromptTemplate.from_template("""
You are an evaluation expert. Evaluate the answer on both faithfulness and relevance.

Question: {question}
Context: {context}
Answer: {answer}

Evaluate on a 0-10 scale:
1. FAITHFULNESS: Is the answer grounded only in the context? (no hallucinations)
2. RELEVANCE: Does the answer directly address the question?

Respond ONLY with:
FAITHFULNESS: [0-10]
RELEVANCE: [0-10]
OVERALL: [average]
VERDICT: [PASS/FAIL - Pass if both >= 8]
EXPLANATION: [one sentence why]
""")

# TODO: Create combined evaluation chain
# Create evaluation chain
combined_chain = combined_prompt | llm | StrOutputParser()


# TODO: Print header (3️⃣ COMBINED EVALUATION)
print("\n3️⃣ COMBINED EVALUATION (Faithfulness + Relevance)")
print("="*70)

# TODO: Loop through all responses and evaluate with combined metrics
# Hint: Invoke chain with question, context, and answer
# Hint: This gives final verdict for each response
for label, response in [("A", response_a), ("B", response_b), ("C", response_c)]:
    result = combined_chain.invoke({
        "question": question,
        "context": context,
        "answer": response
    })
    print(f"\n✨ Response {label}: \"{response}\"")
    print(result)
    print("="*70)



3️⃣ COMBINED EVALUATION (Faithfulness + Relevance)

✨ Response A: "The capital of France is Paris."
FAITHFULNESS: 10  
RELEVANCE: 10  
OVERALL: 10  
VERDICT: PASS  
EXPLANATION: The answer is directly supported by the context and accurately addresses the question without any additional information or errors.

✨ Response B: "The capital of France is Paris, which has a population of 2.1 million."
FAITHFULNESS: 9  
RELEVANCE: 9  
OVERALL: 9  
VERDICT: PASS  
EXPLANATION: The answer accurately reflects the context and directly addresses the question while providing additional relevant information.

✨ Response C: "France is a country in Europe."
FAITHFULNESS: 2  
RELEVANCE: 2  
OVERALL: 2  
VERDICT: FAIL  
EXPLANATION: The answer does not accurately reflect the context or address the question about the capital of France.


### 🎯 Expected Results Summary:

**Response A**: "The capital of France is Paris." → ✅ PASS
- Faithfulness: 10/10 (all info from context)
- Relevance: 10/10 (directly answers)

**Response B**: "The capital of France is Paris, which has a population of 2.1 million." → ❌ FAIL
- Faithfulness: 5/10 (hallucination - population not in context!)
- Relevance: 10/10 (answers question)

**Response C**: "France is a country in Europe." → ❌ FAIL
- Faithfulness: 10/10 (accurate)
- Relevance: 0/10 (doesn't answer what capital is)

### 💡 Key Takeaway:
**Both metrics needed!** Faithfulness catches hallucinations, Relevance catches off-topic answers.

# Part 4: LLM-as-a-Judge Evaluation

## 🤖 Concept:
Use an LLM to automatically evaluate RAG outputs on **multiple dimensions**

## 📊 Why Use LLM-as-a-Judge?
- ✅ **Automated** - No manual grading needed
- ✅ **Fast** - Instant feedback for iteration
- ✅ **Multi-dimensional** - Evaluate relevance, correctness, completeness, helpfulness
- ✅ **Scalable** - Evaluate hundreds of responses

## ⚠️ Considerations:
- LLM judges can make mistakes
- Costs API calls
- May need human validation for critical apps

## 💡 Simple Example: Password Reset Question

**Test Scenario:**
- Question: "How do I reset my password?"
- 3 Answers of different quality levels (Excellent, Poor, Good)

**Judge will score on 4 dimensions:**
1. **Relevance** - Does it address the question?
2. **Correctness** - Is the information accurate?
3. **Completeness** - Does it fully answer?
4. **Helpfulness** - Is it useful and actionable?

In [6]:
# ============================================================================
# STEP 14: Setup Test Data for LLM-as-a-Judge
# ============================================================================
# Recording Hint: Explain the 3 quality levels (poor, good, excellent)

# TODO: Define the test question
question = "How do I reset my password?"

answer1 = "Click 'Forgot Password' on the login page, enter your email, and follow the reset link sent to you."
answer2 = "Reset password."
answer3 = "Click 'Forgot Password'. You'll receive an email with instructions. The reset link expires in 24 hours. Use a strong password with uppercase, lowercase, numbers, and symbols."


# TODO: Define three answers with different quality levels:
# Answer 1: Good answer (covers the basics clearly)
# Answer 2: Poor answer (too brief, unhelpful)
# Answer 3: Excellent answer (comprehensive with security tips)


# TODO: Print the test setup
# Hint: Show question and all 3 answers with quality labels
print("="*70)
print("LLM-AS-A-JUDGE SIMPLE EXAMPLE")
print("="*70)
print(f"\n❓ Question: {question}\n")
print("Answers to evaluate:")
print(f"  1️⃣ {answer1}")
print(f"  2️⃣ {answer2}")
print(f"  3️⃣ {answer3}")
print("\n" + "="*70)

LLM-AS-A-JUDGE SIMPLE EXAMPLE

❓ Question: How do I reset my password?

Answers to evaluate:
  1️⃣ Click 'Forgot Password' on the login page, enter your email, and follow the reset link sent to you.
  2️⃣ Reset password.
  3️⃣ Click 'Forgot Password'. You'll receive an email with instructions. The reset link expires in 24 hours. Use a strong password with uppercase, lowercase, numbers, and symbols.



### Step 1: Create Judge Evaluation Prompt 📝

**Judge scores on 4 criteria (0-10 each):**
- Relevance, Correctness, Completeness, Helpfulness

**Plus**: Overall score & Letter grade (A/B/C/D/F)

In [7]:
# ============================================================================
# STEP 15: Create Judge Evaluation Prompt
# ============================================================================
# Recording Hint: Explain the 4 dimensions and grading scale
judge_prompt = ChatPromptTemplate.from_template("""
You are an expert evaluator. Rate this answer on 4 criteria (0-10 scale each):

Question: {question}
Answer: {answer}

Evaluate on these dimensions:
1. RELEVANCE: Does the answer address the question?
2. CORRECTNESS: Is the information accurate?
3. COMPLETENESS: Does it fully answer the question?
4. HELPFULNESS: Is it useful and actionable?

Respond ONLY in this format:
RELEVANCE: [0-10]
CORRECTNESS: [0-10]
COMPLETENESS: [0-10]
HELPFULNESS: [0-10]
OVERALL: [average of the 4 scores]
GRADE: [A/B/C/D/F based on overall: A=9-10, B=7-8, C=5-6, D=3-4, F=0-2]
FEEDBACK: [one sentence explaining the grade]
""")


# TODO: Create judge prompt that evaluates on 4 criteria
# Hint: Ask for scores on Relevance, Correctness, Completeness, Helpfulness (all 0-10)
# Hint: Also ask for OVERALL (average) and GRADE (A/B/C/D/F)
# Hint: Grading scale: A=9-10, B=7-8, C=5-6, D=3-4, F=0-2
# Hint: Pass {question} and {answer} as template variables



# TODO: Create the judge chain
# Hint: judge_prompt | llm | StrOutputParser()
judge_chain = judge_prompt | llm | StrOutputParser()

# TODO: Print confirmation that chain is created
# Hint: Show the 4 dimensions being evaluated

print("✅ Judge evaluation chain created!")
print("\nThis chain will evaluate answers on 4 dimensions:")
print("  📊 Relevance")
print("  ✅ Correctness")
print("  📝 Completeness")
print("  💡 Helpfulness")


✅ Judge evaluation chain created!

This chain will evaluate answers on 4 dimensions:
  📊 Relevance
  ✅ Correctness
  📝 Completeness
  💡 Helpfulness


### Step 2: Evaluate All Answers ⚖️

Run the judge on all 3 answers and compare scores

In [8]:
# ============================================================================
# STEP 16: Evaluate All Three Answers
# ============================================================================
# Recording Hint: Show how judge scores differ across answer quality levels

# TODO: Print header (JUDGING ALL THREE ANSWERS)
answers = [
    ("Answer 1", answer1),
    ("Answer 2", answer2),
    ("Answer 3", answer3)
]

# TODO: Create list of answers to evaluate
# Hint: [("Answer 1", answer1), ("Answer 2", answer2), ("Answer 3", answer3)]


# TODO: Loop through and evaluate each answer
# Hint: Invoke judge_chain with question and answer
# Hint: Print the answer and evaluation result for each
# Hint: Show how scores differ between Poor (Answer 2), Good (Answer 1), Excellent (Answer 3)

for label, answer in answers:
    print(f"\n{'='*70}")
    print(f"📋 {label}: \"{answer}\"")
    print('='*70)
    
    # Get judge evaluation
    evaluation = judge_chain.invoke({
        "question": question,
        "answer": answer
    })
    
    print(evaluation)

print("\n" + "="*70)
print("✅ All answers evaluated!")
print("="*70)

# TODO: Print completion message



📋 Answer 1: "Click 'Forgot Password' on the login page, enter your email, and follow the reset link sent to you."
RELEVANCE: 10  
CORRECTNESS: 10  
COMPLETENESS: 8  
HELPFULNESS: 9  
OVERALL: 9.25  
GRADE: A  
FEEDBACK: The answer is relevant, accurate, and mostly complete, providing clear and actionable steps for resetting a password.

📋 Answer 2: "Reset password."
RELEVANCE: 5  
CORRECTNESS: 5  
COMPLETENESS: 2  
HELPFULNESS: 3  
OVERALL: 3.75  
GRADE: D  
FEEDBACK: The answer is too vague and does not provide sufficient information or steps to effectively reset a password.

📋 Answer 3: "Click 'Forgot Password'. You'll receive an email with instructions. The reset link expires in 24 hours. Use a strong password with uppercase, lowercase, numbers, and symbols."
RELEVANCE: 10  
CORRECTNESS: 10  
COMPLETENESS: 8  
HELPFULNESS: 9  
OVERALL: 9.25  
GRADE: A  
FEEDBACK: The answer is highly relevant and accurate, providing clear instructions, though it could include more details about wha

### Bonus: Direct Comparison Between Two Answers 🆚

Compare Answer 1 vs Answer 3 directly - which is better?

In [9]:
# ============================================================================
# STEP 17 (BONUS): Direct Comparison Between Two Answers
# ============================================================================
# Recording Hint: Show how to compare answer1 (Good) vs answer3 (Excellent)

# TODO: Create comparison prompt
# Hint: Ask "Which answer is better overall?"
# Hint: Pass {question}, {answer_a}, and {answer_b}
# Hint: Return WINNER, CONFIDENCE, SCORE_A, SCORE_B, and REASON

comparison_prompt = ChatPromptTemplate.from_template("""
You are an expert evaluator. Compare these two answers to the same question.

Question: {question}

Answer A: {answer_a}

Answer B: {answer_b}

Which answer is better overall? Consider relevance, correctness, completeness, and helpfulness.

Respond ONLY in this format:
WINNER: [A or B]
CONFIDENCE: [High/Medium/Low]
SCORE_A: [0-10]
SCORE_B: [0-10]
REASON: [one sentence explaining why the winner is better]
""")

# TODO: Create comparison chain
comparison_chain = comparison_prompt | llm | StrOutputParser()

# TODO: Print header (DIRECT COMPARISON: Answer 1 vs Answer 3)


# TODO: Compare answer1 and answer3
# Hint: Invoke comparison_chain with all three parameters
# Hint: Print the comparison result

print("\n" + "="*70)
print("DIRECT COMPARISON: Answer 1 vs Answer 3")
print("="*70)

result = comparison_chain.invoke({
    "question": question,
    "answer_a": answer1,
    "answer_b": answer3
})

print(f"\n📊 Comparing:")
print(f"   A: \"{answer1}\"")
print(f"   B: \"{answer3}\"")
print("\n" + "-"*70)
print(result)
print("-"*70)
# TODO: Explain the result
# Hint: Show why Answer 3 (with security tips) is likely better than Answer 1



DIRECT COMPARISON: Answer 1 vs Answer 3

📊 Comparing:
   A: "Click 'Forgot Password' on the login page, enter your email, and follow the reset link sent to you."
   B: "Click 'Forgot Password'. You'll receive an email with instructions. The reset link expires in 24 hours. Use a strong password with uppercase, lowercase, numbers, and symbols."

----------------------------------------------------------------------
WINNER: B  
CONFIDENCE: High  
SCORE_A: 7  
SCORE_B: 9  
REASON: Answer B provides additional important information about the expiration of the reset link and tips for creating a strong password, making it more complete and helpful.
----------------------------------------------------------------------


### 🎯 Expected Results Summary:

**Answer 1** (Good): "Click 'Forgot Password'..." → **Grade: A or B**
- Relevance: 10/10, Correctness: 10/10, Completeness: 8/10, Helpfulness: 9/10

**Answer 2** (Poor): "Reset password." → **Grade: F**
- Relevance: 5/10, Correctness: 5/10, Completeness: 1/10, Helpfulness: 1/10

**Answer 3** (Excellent): "Click 'Forgot Password'... (with security tips)" → **Grade: A or A+**
- Relevance: 10/10, Correctness: 10/10, Completeness: 10/10, Helpfulness: 10/10

### 💡 Key Takeaways:

1. **LLM-as-a-Judge** evaluates multiple dimensions automatically
2. **Letter grades (A-F)** make it easy to compare quality
3. **Direct comparison** helps choose between similar answers
4. **Use for**: Automated evaluation, A/B testing, quality monitoring

---